# DEE — Test de cristal elástico: velocidad del sonido y relación de dispersión

**Pregunta:** ¿el sustrato DEE se comporta como un cristal elástico discreto con relación de dispersión fonónica ω(k) bien definida?

**Predicción cristalina:** para un cristal elástico isotrópico, las ondas con vector de onda k se propagan con velocidad del sonido:

$$c_s = \sqrt{\lambda_1 / k_1^2}$$

donde λ₁ es el primer autovalor no trivial del Laplaciano y k₁ es el vector de onda correspondiente. Además, la relación de dispersión debe ser aproximadamente lineal ω(k) = c_s · k para |k| pequeño (régimen acústico).

**Tres tests complementarios:**

1. **Velocidad del sonido** — medir c_s directamente de la propagación de una onda plana
2. **Relación de dispersión** — extraer ω(k) para varios |n|² y ver si es lineal en k
3. **Isotropía** — verificar que c_s es independiente de la dirección del vector de onda

---

In [ ]:
import os
import numpy as np
from scipy.sparse import csr_matrix, diags
from scipy.sparse.linalg import eigsh
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt
import time

PLOT_DIR = 'plots_fonones'
os.makedirs(PLOT_DIR, exist_ok=True)

def save_plot(name, fig=None, dpi=120):
    if fig is None: fig = plt.gcf()
    path = os.path.join(PLOT_DIR, f'{name}.png')
    fig.savefig(path, dpi=dpi, bbox_inches='tight')
    print(f'  → {path}')

print('Setup listo')

In [ ]:
# Funciones base
def periodic_distance_matrix(points, L=1.0):
    diff = points[:, None, :] - points[None, :, :]
    diff = diff - L * np.round(diff / L)
    return np.linalg.norm(diff, axis=-1)

def grid_perturbed_T3(N_target, jitter_fraction, seed=42):
    rng = np.random.default_rng(seed)
    n_side = round(N_target**(1/3))
    N_actual = n_side**3
    spacing = 1.0 / n_side
    coords = np.arange(n_side) * spacing + spacing/2
    gx, gy, gz = np.meshgrid(coords, coords, coords, indexing='ij')
    points = np.stack([gx.ravel(), gy.ravel(), gz.ravel()], axis=1)
    points += rng.uniform(-jitter_fraction*spacing, jitter_fraction*spacing, size=points.shape)
    points = points % 1.0
    return points, N_actual

def build_laplacian(D_mat, k_neighbors=30, alpha=2.0):
    N = len(D_mat)
    D_search = D_mat.copy()
    np.fill_diagonal(D_search, np.inf)
    neighbor_idx = np.argpartition(D_search, k_neighbors, axis=1)[:, :k_neighbors]
    rows, cols, vals = [], [], []
    for i in range(N):
        for j in neighbor_idx[i]:
            d = D_mat[i, j]
            if d > 0:
                rows.append(i); cols.append(j); vals.append(1.0/d**alpha)
    W = csr_matrix((vals, (rows, cols)), shape=(N, N))
    W = (W + W.T) / 2
    degrees = np.array(W.sum(axis=1)).flatten()
    return diags(degrees) - W

# Setup del grafo (el mismo óptimo de SIM 10)
points, N = grid_perturbed_T3(1331, 0.1, seed=42)
D_mat = periodic_distance_matrix(points, L=1.0)
L_op = build_laplacian(D_mat, 30, alpha=2.0)

print(f'N = {N}, grafo construido')

## Paso 1 — Velocidad del sonido desde autovalores del Laplaciano

En un cristal elástico con celda de lado L=1 y modo |n|² = n_x² + n_y² + n_z²:

$$\omega_{|n|^2} = c_s \cdot |k| = c_s \cdot \frac{2\pi}{L} \sqrt{|n|^2}$$

Entonces, de la medición del primer autovalor (|n|² = 1):

$$c_s = \frac{\sqrt{\lambda_1}}{2\pi}$$

In [ ]:
# Calculamos los primeros 50 autovalores para tener varios |n|²
N_EIGS = 50
eigs_all, eigvecs_all = eigsh(L_op, k=N_EIGS, which='SM', tol=1e-8)

# Ordenar
idx_sort = np.argsort(eigs_all)
eigs_all = eigs_all[idx_sort]
eigvecs_all = eigvecs_all[:, idx_sort]

# Filtrar modo cero
eigs_nonzero = eigs_all[eigs_all > 1e-3]
omegas = np.sqrt(eigs_nonzero)

print(f'Primeros {len(eigs_nonzero)} autovalores no-nulos:')
print(f'{"idx":>4} {"λ":>12} {"ω=√λ":>12}')
for i in range(min(20, len(eigs_nonzero))):
    print(f'{i+1:>4} {eigs_nonzero[i]:>12.4f} {omegas[i]:>12.4f}')

# Velocidad del sonido predicha por el primer modo
# En T³ con L=1, el primer modo |n|²=1 tiene |k| = 2π
L_CUBE = 1.0
k1_mag = 2 * np.pi / L_CUBE
c_s_from_first = omegas[0] / k1_mag

print(f'\nVelocidad del sonido (desde primer modo):')
print(f'  ω₁ = {omegas[0]:.4f}')
print(f'  k₁ = 2π/L = {k1_mag:.4f}')
print(f'  c_s = ω₁/k₁ = {c_s_from_first:.4f}')

## Paso 2 — Relación de dispersión ω(|n|)

Para cada autovalor, identificamos su |n|² dominante proyectando el autovector sobre ondas planas. Luego graficamos ω vs |n| para ver si es lineal (régimen acústico cristalino).

In [ ]:
def identify_dominant_mode(eigvec, points, n_max=3):
    """
    Identifica el modo Fourier dominante de un autovector.
    Proyecta sobre cos(2π·n·x) + sin(2π·n·x) para cada n.
    Retorna el n² del modo dominante.
    """
    best_n2 = 0
    best_amp = 0
    best_nvec = None
    
    for nx in range(-n_max, n_max+1):
        for ny in range(-n_max, n_max+1):
            for nz in range(-n_max, n_max+1):
                n2 = nx**2 + ny**2 + nz**2
                if n2 == 0:
                    continue
                n_vec = np.array([nx, ny, nz])
                phase = 2 * np.pi * (points @ n_vec)
                # Proyectamos sobre cos y sin
                amp_cos = np.abs(np.dot(eigvec, np.cos(phase))) / np.sqrt(len(eigvec))
                amp_sin = np.abs(np.dot(eigvec, np.sin(phase))) / np.sqrt(len(eigvec))
                amp = np.sqrt(amp_cos**2 + amp_sin**2)
                
                if amp > best_amp:
                    best_amp = amp
                    best_n2 = n2
                    best_nvec = n_vec
    
    return best_n2, best_amp, best_nvec

# Identificamos los primeros 25 modos
print('Identificación de modos Fourier dominantes:')
print(f'{"idx":>4} {"ω":>10} {"|n|² dominante":>15} {"amplitud":>10} {"n vec":>12}')
print('-'*60)

mode_info = []
for i in range(min(25, len(eigs_nonzero))):
    idx_in_all = np.where(eigs_all == eigs_nonzero[i])[0][0]
    vec = eigvecs_all[:, idx_in_all]
    n2, amp, nvec = identify_dominant_mode(vec, points, n_max=3)
    mode_info.append({'idx': i+1, 'omega': omegas[i], 'n2': n2, 'amp': amp, 'nvec': nvec})
    print(f'{i+1:>4} {omegas[i]:>10.4f} {n2:>15} {amp:>10.3f} {str(nvec):>12}')

In [ ]:
# Agrupamos por |n|² y promediamos ω
from collections import defaultdict
omega_by_n2 = defaultdict(list)
for info in mode_info:
    omega_by_n2[info['n2']].append(info['omega'])

n2_values = sorted(omega_by_n2.keys())
n_mags = np.sqrt(n2_values)
omega_means = np.array([np.mean(omega_by_n2[n2]) for n2 in n2_values])
omega_stds = np.array([np.std(omega_by_n2[n2]) for n2 in n2_values])

print('\nRelación de dispersión ω(|n|):')
print(f'{"|n|²":>6} {"|n|":>8} {"# modos":>8} {"ω medio":>10} {"ω std":>10}')
for n2, mag, mean, std in zip(n2_values, n_mags, omega_means, omega_stds):
    n_count = len(omega_by_n2[n2])
    print(f'{n2:>6} {mag:>8.3f} {n_count:>8} {mean:>10.4f} {std:>10.4f}')

# Ajustar a ley lineal ω = c_s · k = c_s · 2π · |n|
def linear_dispersion(n_mag, cs):
    return cs * 2 * np.pi * n_mag

# Ajuste con peso inverso al error
weights = 1.0 / (omega_stds + 0.01)  # evitamos división por cero
popt, pcov = curve_fit(linear_dispersion, n_mags, omega_means, 
                        sigma=1.0/weights, p0=[c_s_from_first])
cs_fit = popt[0]
cs_err = np.sqrt(pcov[0, 0])

# Residuos
residuals = omega_means - linear_dispersion(n_mags, cs_fit)
rel_residuals = residuals / omega_means * 100

print(f'\nVelocidad del sonido ajustada: c_s = {cs_fit:.4f} ± {cs_err:.4f}')
print(f'Consistencia con primer modo: c_s (primer) = {c_s_from_first:.4f}')
print(f'\nResiduos relativos del ajuste lineal:')
for n2, mag, mean, rel in zip(n2_values, n_mags, omega_means, rel_residuals):
    print(f'  |n|={mag:.3f}: {rel:+.2f}% de ω_predicho')

In [ ]:
# Plot 1: Relación de dispersión
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.errorbar(n_mags, omega_means, yerr=omega_stds, fmt='o', markersize=12,
            color='darkblue', capsize=5, label='Modos DEE')
k_smooth = np.linspace(0, max(n_mags)*1.1, 50)
ax.plot(k_smooth, linear_dispersion(k_smooth, cs_fit), '-',
        color='red', lw=2, label=f'Ajuste lineal c_s = {cs_fit:.3f}')
ax.set_xlabel('|n|', fontsize=12)
ax.set_ylabel('ω', fontsize=12)
ax.set_title('Relación de dispersión ω(|n|) — debería ser lineal si es cristal', fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

# Plot 2: Residuos
ax = axes[1]
ax.plot(n_mags, rel_residuals, 'o-', markersize=12, color='darkgreen', linewidth=2)
ax.axhline(0, color='black', linewidth=1)
ax.fill_between([min(n_mags), max(n_mags)], -5, 5, alpha=0.2, color='green',
                 label='±5% (dispersión lineal OK)')
ax.set_xlabel('|n|', fontsize=12)
ax.set_ylabel('(ω_observado − ω_ajuste) / ω_observado  [%]', fontsize=12)
ax.set_title('Residuos del ajuste lineal', fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

plt.tight_layout()
save_plot('16_relacion_dispersion')
plt.show()

## Paso 3 — Test de isotropía

En un cristal isotrópico (aproximado para T³ homogéneo), la velocidad del sonido debe ser independiente de la dirección del vector de onda k. Los seis modos del sexteto (|n|²=1) corresponden a direcciones ±x̂, ±ŷ, ±ẑ. Si c_s es isotrópica, los seis tienen la misma ω.

In [ ]:
# Extraer los seis modos del sexteto (|n|²=1)
sexteto_omegas = omega_by_n2.get(1, [])

print(f'Modos del sexteto T³ (|n|²=1):')
print(f'  Número de modos: {len(sexteto_omegas)}')
if len(sexteto_omegas) >= 6:
    sexteto_arr = np.array(sexteto_omegas[:6])
    print(f'  ω (los 6): {sexteto_arr}')
    print(f'  Media: {sexteto_arr.mean():.4f}')
    print(f'  Std: {sexteto_arr.std():.4f}')
    isotropia = sexteto_arr.std() / sexteto_arr.mean() * 100
    print(f'  Isotropía (std/media): {isotropia:.2f}%')
    
    if isotropia < 2:
        print(f'  → Cristal isótropo (muy buena isotropía)')
    elif isotropia < 5:
        print(f'  → Cristal casi-isótropo (isotropía aceptable)')
    else:
        print(f'  → Anisotropía significativa (revisar)')
else:
    print(f'  No hay suficientes modos |n|²=1 identificados')

## Paso 4 — Propagación de onda plana: velocidad directa

Validación independiente: integramos $\ddot\phi = -L\phi$ con condición inicial onda plana |n|=1 y medimos directamente cuánto tarda una perturbación en recorrer una distancia conocida.

In [ ]:
# Condición inicial: onda plana con |n|=1 en dirección x
n_vec = np.array([1, 0, 0])
phase = 2 * np.pi * (points @ n_vec)
phi_init = np.cos(phase)

# Proyector sobre el modo (1,0,0)
basis_cos = np.cos(phase) / np.sqrt(N)
basis_sin = np.sin(phase) / np.sqrt(N)

# Integración leapfrog
T_TOTAL = 10.0
DT = 0.005
N_STEPS = int(T_TOTAL / DT)

phi = phi_init.copy()
phi_dot = np.zeros(N)
amp_cos_history = np.zeros(N_STEPS)
amp_sin_history = np.zeros(N_STEPS)
times = np.arange(N_STEPS) * DT

t0 = time.time()
for step in range(N_STEPS):
    acc = -(L_op @ phi)
    phi_dot += acc * DT
    phi += phi_dot * DT
    amp_cos_history[step] = np.dot(phi, basis_cos) * np.sqrt(N)
    amp_sin_history[step] = np.dot(phi, basis_sin) * np.sqrt(N)

print(f'Integración: {time.time()-t0:.1f}s')

# La amplitud total del modo (1,0,0) es √(cos² + sin²)
amp_total = np.sqrt(amp_cos_history**2 + amp_sin_history**2)

# Plot
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(times, amp_cos_history, '-', color='steelblue', label='cos(ωt)', lw=1)
ax.plot(times, amp_sin_history, '-', color='darkorange', label='sin(ωt)', lw=1, alpha=0.7)
ax.set_xlabel('Tiempo t', fontsize=12)
ax.set_ylabel('Amplitud del modo (1,0,0)', fontsize=12)
ax.set_title('Evolución temporal del modo (1,0,0) — debería ser oscilación pura sin decaimiento', fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
save_plot('17_oscilacion_pura')
plt.show()

# Extraer ω por FFT
from scipy.fft import rfft, rfftfreq
signal = amp_cos_history - amp_cos_history.mean()
fft_signal = np.abs(rfft(signal))
freqs = rfftfreq(len(signal), DT)
omegas_fft = 2 * np.pi * freqs
peak_idx = np.argmax(fft_signal)
omega_measured = omegas_fft[peak_idx]
cs_measured = omega_measured / (2 * np.pi)

print(f'\nMedición directa por propagación:')
print(f'  ω observada = {omega_measured:.4f}')
print(f'  c_s medida = {cs_measured:.4f}')
print(f'  c_s desde autovalores = {c_s_from_first:.4f}')
print(f'  Diferencia: {abs(cs_measured - c_s_from_first) / c_s_from_first * 100:.2f}%')

## Paso 5 — Veredicto

In [ ]:
print('='*70)
print('VEREDICTO — ¿El sustrato DEE es un cristal elástico?')
print('='*70)
print()

# Criterio 1: linealidad de dispersión
max_resid = np.max(np.abs(rel_residuals))
linearity_ok = max_resid < 15
print(f'Criterio 1 - Dispersión lineal ω(|n|) = c_s · 2π · |n|:')
print(f'  Residuo máximo: {max_resid:.2f}%')
print(f'  → {"PASA" if linearity_ok else "FALLA"} (umbral 15%)')
print()

# Criterio 2: isotropía del sexteto
if len(sexteto_omegas) >= 6:
    sexteto_arr = np.array(sexteto_omegas[:6])
    isotropy_val = sexteto_arr.std() / sexteto_arr.mean() * 100
    isotropy_ok = isotropy_val < 5
    print(f'Criterio 2 - Isotropía del sexteto (|n|²=1):')
    print(f'  Dispersión: {isotropy_val:.2f}%')
    print(f'  → {"PASA" if isotropy_ok else "FALLA"} (umbral 5%)')
else:
    isotropy_ok = False
    print(f'Criterio 2 - Isotropía: no hay suficientes modos del sexteto')
print()

# Criterio 3: consistencia entre autovalores y propagación
propagation_ok = abs(cs_measured - c_s_from_first) / c_s_from_first < 0.05
print(f'Criterio 3 - Consistencia entre autovalores y propagación:')
print(f'  c_s (autovalores): {c_s_from_first:.4f}')
print(f'  c_s (propagación): {cs_measured:.4f}')
print(f'  Diferencia: {abs(cs_measured - c_s_from_first) / c_s_from_first * 100:.2f}%')
print(f'  → {"PASA" if propagation_ok else "FALLA"} (umbral 5%)')
print()

# Veredicto global
n_pass = sum([linearity_ok, isotropy_ok, propagation_ok])
print('='*70)
if n_pass == 3:
    print('\n✓✓✓ MAPEO CRISTALINO CONFIRMADO')
    print()
    print('El sustrato DEE es matemáticamente equivalente a un cristal elástico discreto.')
    print(f'Velocidad del sonido: c_s = {cs_fit:.4f} ± {cs_err:.4f}')
    print('Relación de dispersión lineal en régimen acústico.')
    print('Isotropía consistente con T³ homogéneo.')
    print()
    print('Toda la maquinaria de física de sólidos cristalinos es aplicable:')
    print('- Teoría fonónica (ramas acústica/óptica)')
    print('- Teorema de Bloch y zonas de Brillouin')
    print('- Densidad de estados de Debye')
    print('- Análisis de defectos (vacancias, dislocaciones)')
    print('- Relaciones termodinámicas (calor específico, conductividad)')
elif n_pass == 2:
    print('\n~ MAPEO CRISTALINO PARCIAL')
    print('Algunos tests pasan pero hay matices a investigar.')
else:
    print('\n✗ MAPEO CRISTALINO DESAFIADO')
    print('El sustrato no se comporta estrictamente como cristal elástico.')
print('='*70)

print(f'\nScore: {n_pass}/3')

In [ ]:
import shutil
shutil.make_archive('plots_fonones', 'zip', PLOT_DIR)
print(f'\nZip creado: plots_fonones.zip')

try:
    from google.colab import files
    files.download('plots_fonones.zip')
    print('→ Descarga iniciada')
except ImportError:
    pass